# SDK source inspection notebook for Fabric Data Agent evaluation

This notebook collects the Python snippets used to inspect the `fabric-data-agent-sdk` source code directly from a notebook environment.

Its purpose is not to evaluate the Data Agent itself, but to understand how the SDK behaves internally: where evaluation results are stored, how the summary is built, which model appears to be used in different parts of the evaluation flow, how the default critic prompt is implemented, and whether placeholders such as `{actual_answer}` are really supported at runtime.


## 1. Install the latest Fabric Data Agent SDK

We start by installing or upgrading the SDK inside the notebook kernel.

This ensures that the source code we inspect corresponds to the exact package version currently available in the environment, rather than to an older cached version.


In [1]:
%pip install -U fabric-data-agent-sdk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of azure-kusto-data to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of azure-kusto-data to determine which version is compatible with other requirements. This could take a while.
Reason for being yanked: Yanked due to conflicts with CVE-2024-35195 mitigation
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 699.3/699.3 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51

## 2. Locate and inspect the default critic prompt in the SDK source

This cell scans all Python files inside the installed `fabric.dataagent` package and looks for references to `critic_prompt`.

The goal is to extract the built-in evaluation prompt used when no custom `critic_prompt` is provided, and to inspect the surrounding code that handles critic prompt generation and usage.


In [5]:
import fabric.dataagent as fd
from pathlib import Path

package_root = Path(fd.__file__).resolve().parent

for py_file in package_root.rglob("*.py"):
    try:
        lines = py_file.read_text(encoding="utf-8").splitlines()
    except Exception:
        continue

    hits = [i for i, line in enumerate(lines) if "critic_prompt" in line]

    if hits:
        print("=" * 120)
        print("FILE:", py_file)
        for idx in hits:
            start = max(0, idx - 15)
            end = min(len(lines), idx + 35)
            print("-" * 120)
            for j in range(start, end):
                print(f"{j+1:04d}: {lines[j]}")
            print()

FILE: /home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/fabric/dataagent/evaluation/_models.py
------------------------------------------------------------------------------------------------------------------------
0090: 
0091: class EvaluationContext(BaseModel):
0092:     """
0093:     Input parameters for data agent evaluation.
0094:     
0095:     Attributes
0096:     ----------
0097:     row : dict
0098:         A single row from the input DataFrame containing question and expected_answer.
0099:     data_agent_name : str
0100:         Name of the Data Agent.
0101:     workspace_name : str, optional
0102:         Workspace Name if Data Agent is in different workspace.
0103:     table_name : str
0104:         Table name to store the evaluation result.
0105:     critic_prompt : str, optional
0106:         Prompt to evaluate the actual answer from Data Agent.
0107:     data_agent_stage : str
0108:         Data Agent stage ie., sandbox or production.
0109: 

## 3. Inspect how evaluation outputs are persisted

This cell looks inside the internal `_storage` module and extracts source code for selected helper functions.

The key purpose is to understand where the SDK writes the evaluation output tables, how it resolves the default Lakehouse path, and how data is loaded or persisted during the evaluation workflow.


In [3]:
import inspect
import importlib

mod = importlib.import_module("fabric.dataagent.evaluation._storage")

for fn_name in ["_save_output", "_default_lakehouse_path", "_get_data"]:
    fn = getattr(mod, fn_name, None)
    print("=" * 120)
    print(f"FUNCTION: {fn_name}")

    if fn is None:
        print("Non trovata")
        continue

    try:
        print(inspect.getsource(fn))
    except Exception as e:
        print(f"Errore nell'estrazione del sorgente: {e!r}")

FUNCTION: _save_output
def _save_output(df: pd.DataFrame, table_name: str):
    """
    Saves the Dataframe to the Delta table.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with processed output rows.
    table_name : str
        Table name to store the evaluation result
    """
    try:
        lakehouse_path = _default_lakehouse_path()
        table_path = f"{lakehouse_path}/Tables/{table_name}"

        if _on_jupyter():
            try:
                from deltalake.writer import write_deltalake
                write_deltalake(table_path, df, mode="append")
            except ImportError:
                logging.error("deltalake module not found. Please install it to use this feature.")
            except Exception as e:
                logging.error(f"Error writing to Delta table: {str(e)}")
        else:
            try:
                from pyspark.sql import SparkSession
                from pyspark.sql.functions import col

                spark = Sp

## 4. Inspect the implementation of `get_evaluation_summary`

This snippet extracts the source code of the public `get_evaluation_summary` function.

It is useful when we want to verify exactly how the high-level summary is built, which columns are counted, and whether the aggregation logic matches what we see in the notebook output.


In [2]:
import inspect
import importlib

mod = importlib.import_module("fabric.dataagent.evaluation.report")
fn = getattr(mod, "get_evaluation_summary", None)

if fn is None:
    print("Funzione get_evaluation_summary non trovata")
else:
    print(f"Modulo: {mod.__file__}")
    print("=" * 120)
    print(inspect.getsource(fn))

Modulo: /home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/fabric/dataagent/evaluation/report.py
def get_evaluation_summary(
    table_name: str = 'evaluation_output', verbose: bool = False
):
    """
    Overall summary of an evaluation stored in the delta table.

    Parameters
    ----------
    table_name : str, optional
        Table name to store the evaluation result. Default to 'evaluation_output'.
    verbose : bool, optional
        Flag to print the evaluation summary. Default to False.

    Returns
    -------
    pd.DataFrame
        DataFrame with summary details.
    """

    # Get the delta table
    df = _get_data(table_name)

    # Return if table is not found
    if df is None:
        return None

    # Calculate the percentage of True values in the DataFrame
    eval_percentage = (df['evaluation_judgement'].mean()) * 100

    # Group by timestamp and count the occurrences of True, False, and None in the 'evaluation_judgement' column
    

## 5. Check the installed package version and environment

This optional cell lists installed Python packages in the notebook environment.

It is useful to confirm which version of `fabric-data-agent-sdk` is actually installed before or after upgrade, and to document the environment used for source-code inspection.


In [1]:
%pip list

Package                            Version
---------------------------------- --------------------
absl-py                            2.3.1
adlfs                              2023.4.0
aiohappyeyeballs                   2.6.1
aiohttp                            3.11.11
aiosignal                          1.4.0
alembic                            1.16.5
altair                             5.4.1
annotated-types                    0.7.0
anyio                              4.10.0
archspec                           0.2.5
argon2-cffi                        25.1.0
argon2-cffi-bindings               25.1.0
arrow                              1.3.0
astor                              0.8.1
asttokens                          3.0.0
async-lru                          2.0.5
atpublic                           5.1
attrs                              25.3.0
azure-common                       1.1.28
azure-core                         1.29.4
azure-datalake-store               0.0.53
azure-identity               

## 6. Search the SDK source for model references

This broader scan searches the SDK for references to model names and model-related helpers, including strings such as `gpt-4.1`, `gpt-4o`, `FABRIC_DEFAULT_MODEL`, and selected client calls.

The goal is to understand which model appears to be used in the evaluation path and which one is referenced in adjacent utilities such as few-shot validation.


In [2]:
import fabric.dataagent as fd
from pathlib import Path
import re

package_root = Path(fd.__file__).resolve().parent
print("Package root:", package_root)
print()

patterns = [
    r"gpt-4\.1",
    r"gpt-4o",
    r"gpt-5",
    r"FABRIC_DEFAULT_MODEL",
    r"model\s*=",
    r"chat\.completions\.create",
    r"responses\.create",
    r"FabricOpenAI",
    r"_get_message\(",
    r"evaluate_data_agent\(",
]

for py_file in package_root.rglob("*.py"):
    try:
        text = py_file.read_text(encoding="utf-8")
    except Exception:
        continue

    hits = []
    for pat in patterns:
        for m in re.finditer(pat, text, flags=re.IGNORECASE):
            hits.append((pat, m.start()))

    if hits:
        print("=" * 120)
        print("FILE:", py_file)
        for pat, pos in hits[:20]:
            start = max(0, pos - 300)
            end = min(len(text), pos + 1200)
            print("-" * 120)
            print("PATTERN:", pat)
            print(text[start:end])
            print()

Package root: /home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/fabric/dataagent

FILE: /home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/fabric/dataagent/client/_fabric_openai.py
------------------------------------------------------------------------------------------------------------------------
PATTERN: FabricOpenAI
t-scenario"
AI_SKILL_STAGE_HEADER_NAME = "x-ms-ai-aiskill-stage"

AI_SKILL_STAGE_T = t.Literal["sandbox", "production", None]
ASSISTANT_SCENARIOS_T = t.Literal["aiskill", "dscopilot", "dwcopilot", "aiskillv1shim"]
FEATURE_NAMES_T: t.TypeAlias = t.Literal["AISkill", "DSCopilot", "DWCopilot"]


class FabricOpenAI(OpenAI):
    """
    Interact with Fabric's AI Assistant API.

    Parameters
    ----------
    artifact_name : str
        The name of the Data Agent artifact.
    workspace_name : str, optional
        The name of the workspace. If not provided, it defaults to the notebook's workspace name.
    api_ver

## 7. Inspect the evaluation thread message path

This cell opens the internal `_thread` module and extracts the `_get_message` function together with the `OPEN_AI_MODEL` constant.

This is the most direct way to verify which model the SDK appears to use in the evaluation conversation path.


In [3]:
import inspect
import importlib

mod = importlib.import_module("fabric.dataagent.evaluation._thread")

print("MODULE:", mod.__file__)
print("=" * 120)
print(inspect.getsource(mod._get_message))
print("=" * 120)
print("OPEN_AI_MODEL =", getattr(mod, "OPEN_AI_MODEL", None))

MODULE: /home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/fabric/dataagent/evaluation/_thread.py
def _get_message(fabric_client: FabricOpenAI, thread_id: str, query: str):
    """
    Get message for the input query and thread.

    Parameters
    ----------
    fabric_client: FabricOpenAI
        An instance of the fabric client created to interact with Data Agent.
    thread_id: str
        An unique identifier of the thread.
    query : str
        Question from the input DataFrame.

    Returns
    -------
    tuple[str, object]
        Tuple with actual answer and run instance.

    Raises
    -------
        RuntimeError: If fabric client is None.
    """

    import os

    # Raise RuntimeError if fabric client is None
    if fabric_client is None:
        logging.debug("Fabric client is None")
        raise RuntimeError("Fabric client is None")

    message_ids = []

    try:
        # Create assistant
        assistant = fabric_client.beta.assistan

## 8. Inspect the few-shot validation utility separately

This cell inspects the `_few_shot_validation` module.

It is useful because it lets us distinguish the model used in the main evaluation flow from the model referenced in the internal utility that validates few-shot examples. That distinction matters when comparing GPT-4o and GPT-4.1 references in the SDK.


In [4]:
import inspect
import importlib

mod = importlib.import_module("fabric.dataagent.evaluation._few_shot_validation")

print("MODULE:", mod.__file__)
print("FABRIC_DEFAULT_MODEL =", mod.FABRIC_DEFAULT_MODEL)
print()
print(inspect.getsource(mod._evaluate_few_shot_examples))

MODULE: /home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/fabric/dataagent/evaluation/_few_shot_validation.py
FABRIC_DEFAULT_MODEL = gpt-4.1

def _evaluate_few_shot_examples(
    examples: list[dict[str, str]],
    datasource_type: str,
    batch_size: int = 10
) -> FewShotEvalResult:
    """
    Internal utility function to evaluate few-shot examples using an LLM model.
    
    NOTE: SQL-only. This function is designed for SQL queries (Lakehouse/Warehouse).
    It is not suitable for KQL (KustoDb), Ontology, or DAX (Semantic Models) yet.
    
    NOTE: This is an internal utility function. Users should call datasource.evaluate_few_shots()
    instead, which provides a cleaner API and returns results as DataFrames.

    Args:
        examples: List of dicts with 'natural language' and 'sql' keys.
        datasource_type: Type of datasource (e.g., 'lakehouse', 'warehouse').
        batch_size: Number of examples per batch.

    Returns:
        FewShotEvalR

## 9. Verify whether `{actual_answer}` is really supported in custom critic prompts

This final scan searches the SDK source for references to `actual_answer`, `{actual_answer}`, prompt formatting calls, and critic-prompt generation logic.

The objective is to verify whether `actual_answer` is truly injected into custom `critic_prompt` templates at runtime, or whether it only exists in stored evaluation results and examples found in external documentation.


In [2]:
import fabric.dataagent as fd
from pathlib import Path
import re

package_root = Path(fd.__file__).resolve().parent
print("Package root:", package_root)
print()

patterns = [
    r"actual_answer",
    r"\{actual_answer\}",
    r"\.format\(",
    r"critic_prompt",
    r"_generate_prompt",
]

for py_file in package_root.rglob("*.py"):
    try:
        text = py_file.read_text(encoding="utf-8")
    except Exception:
        continue

    hits = []
    for pat in patterns:
        for m in re.finditer(pat, text, flags=re.IGNORECASE):
            hits.append((pat, m.start()))

    if hits:
        print("=" * 120)
        print("FILE:", py_file)

        # dedup approssimativa per posizione vicina
        shown = []
        for pat, pos in sorted(hits, key=lambda x: x[1]):
            if any(abs(pos - s) < 200 for s in shown):
                continue
            shown.append(pos)

            start = max(0, pos - 500)
            end = min(len(text), pos + 1500)

            print("-" * 120)
            print("PATTERN:", pat)
            print(text[start:end])
            print()

Package root: /home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/fabric/dataagent

FILE: /home/trusted-service-user/jupyter-env/python3.11/lib/python3.11/site-packages/fabric/dataagent/evaluation/_models.py
------------------------------------------------------------------------------------------------------------------------
PATTERN: critic_prompt
 None
    kql: Optional[str] = None


class EvaluationContext(BaseModel):
    """
    Input parameters for data agent evaluation.
    
    Attributes
    ----------
    row : dict
        A single row from the input DataFrame containing question and expected_answer.
    data_agent_name : str
        Name of the Data Agent.
    workspace_name : str, optional
        Workspace Name if Data Agent is in different workspace.
    table_name : str
        Table name to store the evaluation result.
    critic_prompt : str, optional
        Prompt to evaluate the actual answer from Data Agent.
    data_agent_stage : str
  